**Step -1: Environment Setup (Colab or local -- auto-detected)**

Works unmodified in Google Colab or a local Jupyter/VS Code kernel. In
Colab, it mounts Google Drive and moves into a persistent project folder
there, so the ~2.4GB dataset and your trained model survive across Colab
sessions instead of being wiped when the runtime disconnects. Locally, it
just uses the notebook's own folder -- nothing to mount.

In [6]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    PROJECT_DIR = '/content/drive/MyDrive/icece2026_tinyml'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f"Running in Colab -- working directory: {PROJECT_DIR}")
else:
    print("Running locally -- using the notebook's own folder as the working directory.")

Running locally -- using the notebook's own folder as the working directory.


**Step 0: Environment Check**

Confirms the virtual environment has everything installed and reports
whether TensorFlow sees a GPU. A GPU is *not* required — DS-CNN-S is a
~24K-parameter model, so CPU-only training is perfectly fine and is what
most local setups will use. This cell should run without errors before you
go any further; if `import tensorflow` fails here, fix that first (see the
setup instructions) rather than debugging it after the dataset download.

In [7]:
import sys
print(f"Python version: {sys.version}")

import numpy as np
print(f"NumPy version:      {np.__version__}")

import scipy
print(f"SciPy version:      {scipy.__version__}")

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) detected: {[g.name for g in gpus]}")
else:
    print("No GPU detected -- running on CPU. This is fine for DS-CNN-S "
          "(~24K parameters); training will just take a bit longer than on "
          "a GPU machine.")

Python version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
NumPy version:      2.2.2
SciPy version:      1.15.1
TensorFlow version: 2.21.0
No GPU detected -- running on CPU. This is fine for DS-CNN-S (~24K parameters); training will just take a bit longer than on a GPU machine.


**Download & Extract the Dataset**

In [9]:
import os
import urllib.request
import tarfile
import glob

DATASET_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
DATA_DIR = "./speech_commands_v2"
TAR_PATH = "speech_commands_v0.02.tar.gz"

# Use a file that only exists after a *complete* extraction as the
# "are we done" check, instead of just the directory's existence -- a
# partially-downloaded or partially-extracted attempt still creates the
# directory, so checking the directory alone can't tell a broken dataset
# from a good one.
MARKER_FILE = os.path.join(DATA_DIR, "validation_list.txt")

if not os.path.exists(MARKER_FILE):
    os.makedirs(DATA_DIR, exist_ok=True)

    if not os.path.exists(TAR_PATH):
        print("Downloading Speech Commands v2 (~2.4 GB)...")
        urllib.request.urlretrieve(DATASET_URL, TAR_PATH)
        print("Download complete.")
    else:
        print(f"Found existing archive at {TAR_PATH} -- skipping download.")

    print("Extracting archive...")
    try:
        with tarfile.open(TAR_PATH, "r:gz") as tar:
            tar.extractall(DATA_DIR)
        print("Extraction complete!")
    except (tarfile.ReadError, EOFError) as e:
        print(f"Extraction failed ({e}) -- the archive is likely incomplete/corrupted.")
        print(f"Fix: delete '{TAR_PATH}' and re-run this cell to re-download it.")
        raise
else:
    print("Dataset already present and verified (validation_list.txt found) -- skipping.")

# Verify the extraction is actually complete before moving on, rather than
# finding out two cells later with a confusing FileNotFoundError.
required = ["validation_list.txt", "testing_list.txt", "_background_noise_"]
missing = [r for r in required if not os.path.exists(os.path.join(DATA_DIR, r))]
if missing:
    raise RuntimeError(
        f"Dataset looks incomplete -- missing {missing} inside {DATA_DIR}. "
        f"Delete the '{DATA_DIR}' folder (and '{TAR_PATH}' if present) and re-run this cell."
    )

wav_count = len(glob.glob(os.path.join(DATA_DIR, "*", "*.wav")))
print(f"Verified: found {wav_count} .wav files across word folders "
      f"(expected roughly 105,000).")

Found existing archive at speech_commands_v0.02.tar.gz -- skipping download.
Extracting archive...


<HOME>\AppData\Local\Temp\ipykernel_21180\1025802177.py:30: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATA_DIR)


Extraction complete!
Verified: found 105835 .wav files across word folders (expected roughly 105,000).


**Step 2: Build the Official 12-Class Dataset Split**

In [10]:
import glob
from pathlib import Path

# The 10 standard benchmark target words
TARGET_WORDS = {'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go'}
DATASET_PATH = Path("./speech_commands_v2")

# Read official validation and testing file lists
with open(DATASET_PATH / "validation_list.txt", "r") as f:
    val_files = set(f.read().splitlines())

with open(DATASET_PATH / "testing_list.txt", "r") as f:
    test_files = set(f.read().splitlines())

train_samples, val_samples, test_samples = [], [], []

# Iterate over all wav files (excluding background noise folder)
all_wavs = glob.glob(str(DATASET_PATH / "*/*.wav"))

for wav_path in all_wavs:
    rel_path = os.path.relpath(wav_path, DATASET_PATH).replace("\\", "/")

    if "_background_noise_" in rel_path:
        continue

    word = rel_path.split("/")[0]
    label = word if word in TARGET_WORDS else "_unknown_"

    sample_entry = (wav_path, label)

    if rel_path in test_files:
        test_samples.append(sample_entry)
    elif rel_path in val_files:
        val_samples.append(sample_entry)
    else:
        train_samples.append(sample_entry)

print(f"Dataset Split Counts:")
print(f"  Training samples:   {len(train_samples)}")
print(f"  Validation samples: {len(val_samples)}")
print(f"  Testing samples:    {len(test_samples)}")

Dataset Split Counts:
  Training samples:   84843
  Validation samples: 9981
  Testing samples:    11005


**Step 3: Balance the `_unknown_` Class and Synthesize `_silence_` Samples**

The split above has two gaps relative to the standard 12-class Speech Commands
benchmark (10 keywords + `_unknown_` + `_silence_`) that the Methodology
section commits us to:

1. **`_unknown_` is currently every non-keyword word, unbalanced.** There are
   ~25 auxiliary words (bed, bird, cat, ...), each with roughly as many
   samples as a target keyword, so `_unknown_` is likely 15-25x larger than
   any single keyword class. Left as-is, this would bias training toward
   predicting "unknown" and make our accuracy not comparable to the
   literature's 12-class figures.
2. **`_silence_` is missing entirely** — the code above explicitly skips the
   `_background_noise_` folder, so right now we have an *11*-class problem,
   not 12.

This step fixes both: it downsamples `_unknown_` per split to roughly one
keyword-class worth of samples, and synthesizes `_silence_` clips by taking
random 1-second crops of the background-noise recordings (a standard
practice in the KWS literature, e.g. the original TensorFlow Speech
Commands recipe and Hello Edge). Both use fixed random seeds for
reproducibility.

In [11]:
import random
import numpy as np
from scipy.io import wavfile

random.seed(42)
np.random.seed(42)

SAMPLE_RATE = 16000
CLIP_SAMPLES = SAMPLE_RATE  # 1-second clips

# --- 3a. Separate target-word samples from the pooled "_unknown_" samples ---
def split_target_unknown(samples):
    target = [s for s in samples if s[1] != "_unknown_"]
    unknown = [s for s in samples if s[1] == "_unknown_"]
    return target, unknown

train_target, train_unknown = split_target_unknown(train_samples)
val_target, val_unknown     = split_target_unknown(val_samples)
test_target, test_unknown   = split_target_unknown(test_samples)

def avg_target_count(target_list):
    counts = {}
    for _, label in target_list:
        counts[label] = counts.get(label, 0) + 1
    return int(round(sum(counts.values()) / len(counts))), counts

train_avg, train_counts = avg_target_count(train_target)
val_avg, val_counts     = avg_target_count(val_target)
test_avg, test_counts   = avg_target_count(test_target)

print("Per-keyword sample counts (train):", train_counts)
print(f"Average per-keyword count -> train:{train_avg}  val:{val_avg}  test:{test_avg}")
print(f"Pooled '_unknown_' before downsampling -> train:{len(train_unknown)} "
      f"val:{len(val_unknown)} test:{len(test_unknown)}")

# Downsample "_unknown_" to roughly one keyword-class worth of samples per
# split, so it behaves like just another class instead of dominating training.
random.shuffle(train_unknown); random.shuffle(val_unknown); random.shuffle(test_unknown)
train_unknown_ds = train_unknown[:train_avg]
val_unknown_ds   = val_unknown[:val_avg]
test_unknown_ds  = test_unknown[:test_avg]

print(f"'_unknown_' after downsampling  -> train:{len(train_unknown_ds)} "
      f"val:{len(val_unknown_ds)} test:{len(test_unknown_ds)}")

Per-keyword sample counts (train): {'left': 3037, 'no': 3130, 'yes': 3228, 'go': 3106, 'down': 3134, 'off': 2970, 'right': 3019, 'stop': 3111, 'up': 2948, 'on': 3086}
Average per-keyword count -> train:3077  val:370  test:407
Pooled '_unknown_' before downsampling -> train:54074 val:6278 test:6931
'_unknown_' after downsampling  -> train:3077 val:370 test:407


In [12]:
# --- 3b. Synthesize "_silence_" clips from the background-noise recordings ---
BG_NOISE_DIR = DATASET_PATH / "_background_noise_"
SILENCE_DIR = DATASET_PATH / "_silence_generated_"
SILENCE_DIR.mkdir(exist_ok=True)

bg_files = sorted(glob.glob(str(BG_NOISE_DIR / "*.wav")))
print(f"Found {len(bg_files)} background-noise source files.")

bg_waveforms = []
for f in bg_files:
    sr, data = wavfile.read(f)
    assert sr == SAMPLE_RATE, f"Unexpected sample rate in {f}: {sr}"
    bg_waveforms.append(data.astype(np.float32) / 32768.0)

def synth_silence_clips(n_clips, split_name, rng):
    out_dir = SILENCE_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    samples = []
    for i in range(n_clips):
        wav = bg_waveforms[rng.randrange(len(bg_waveforms))]
        max_start = max(len(wav) - CLIP_SAMPLES, 1)
        start = rng.randrange(0, max_start)
        clip = wav[start:start + CLIP_SAMPLES]
        if len(clip) < CLIP_SAMPLES:
            clip = np.pad(clip, (0, CLIP_SAMPLES - len(clip)))
        out_path = out_dir / f"silence_{i:05d}.wav"
        wavfile.write(str(out_path), SAMPLE_RATE, (clip * 32767).astype(np.int16))
        samples.append((str(out_path), "_silence_"))
    return samples

# Separate RNGs per split so train/val/test don't draw identical crops
rng_train, rng_val, rng_test = random.Random(42), random.Random(43), random.Random(44)
train_silence = synth_silence_clips(train_avg, "train", rng_train)
val_silence   = synth_silence_clips(val_avg, "val", rng_val)
test_silence  = synth_silence_clips(test_avg, "test", rng_test)

print(f"Synthesized silence clips -> train:{len(train_silence)} "
      f"val:{len(val_silence)} test:{len(test_silence)}")

Found 6 background-noise source files.


<HOME>\AppData\Local\Temp\ipykernel_21180\1450764636.py:11: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, data = wavfile.read(f)


Synthesized silence clips -> train:3077 val:370 test:407


In [13]:
# --- 3c. Assemble the final, balanced 12-class splits ---
train_samples_balanced = train_target + train_unknown_ds + train_silence
val_samples_balanced   = val_target + val_unknown_ds + val_silence
test_samples_balanced  = test_target + test_unknown_ds + test_silence

random.shuffle(train_samples_balanced)
random.shuffle(val_samples_balanced)
random.shuffle(test_samples_balanced)

LABELS = ['_silence_', '_unknown_', 'yes', 'no', 'up', 'down',
          'left', 'right', 'on', 'off', 'stop', 'go']
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}

def print_class_distribution(samples, name):
    counts = {l: 0 for l in LABELS}
    for _, label in samples:
        counts[label] += 1
    print(f"\n{name} ({len(samples)} samples total):")
    for l in LABELS:
        print(f"  {l:10s}: {counts[l]}")

print_class_distribution(train_samples_balanced, "Train")
print_class_distribution(val_samples_balanced, "Validation")
print_class_distribution(test_samples_balanced, "Test")


Train (36923 samples total):
  _silence_ : 3077
  _unknown_ : 3077
  yes       : 3228
  no        : 3130
  up        : 2948
  down      : 3134
  left      : 3037
  right     : 3019
  on        : 3086
  off       : 2970
  stop      : 3111
  go        : 3106

Validation (4443 samples total):
  _silence_ : 370
  _unknown_ : 370
  yes       : 397
  no        : 406
  up        : 350
  down      : 377
  left      : 352
  right     : 363
  on        : 363
  off       : 373
  stop      : 350
  go        : 372

Test (4888 samples total):
  _silence_ : 407
  _unknown_ : 407
  yes       : 419
  no        : 405
  up        : 425
  down      : 406
  left      : 412
  right     : 396
  on        : 396
  off       : 402
  stop      : 411
  go        : 402


**Step 4: MFCC Feature Extraction Pipeline**

Standard KWS front end: 40 ms frames, 20 ms stride, 40 MFCC coefficients,
computed entirely in TensorFlow ops (`tf.signal`) so it runs on GPU and
stays inside a `tf.data` graph. For a 1-second (16,000-sample) clip this
produces a `(49, 40)` feature map — 49 time frames x 40 MFCC coefficients —
matching what's stated in the Methodology section.

In [14]:
import tensorflow as tf

FRAME_LENGTH = 640   # 40 ms at 16 kHz
FRAME_STEP   = 320   # 20 ms at 16 kHz
FFT_LENGTH   = 1024
NUM_MFCC     = 40
NUM_MEL_BINS = 40

def load_wav(path):
    audio_binary = tf.io.read_file(path)
    waveform, _ = tf.audio.decode_wav(audio_binary, desired_channels=1,
                                       desired_samples=CLIP_SAMPLES)
    return tf.squeeze(waveform, axis=-1)

def compute_mfcc(waveform):
    stft = tf.signal.stft(waveform, frame_length=FRAME_LENGTH,
                           frame_step=FRAME_STEP, fft_length=FFT_LENGTH)
    spectrogram = tf.abs(stft)
    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=NUM_MEL_BINS,
        num_spectrogram_bins=spectrogram.shape[-1],
        sample_rate=SAMPLE_RATE, lower_edge_hertz=20.0, upper_edge_hertz=4000.0)
    mel_spectrogram = tf.tensordot(spectrogram, mel_weight_matrix, 1)
    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)
    mfccs = tf.signal.mfccs_from_log_mel_spectrograms(log_mel_spectrogram)
    return mfccs[..., :NUM_MFCC]

# Sanity check on one real sample
sample_path, sample_label = train_samples_balanced[0]
sample_wave = load_wav(sample_path)
sample_mfcc = compute_mfcc(sample_wave)
print(f"Sample label: {sample_label}")
print(f"Waveform shape: {sample_wave.shape}")
print(f"MFCC feature shape: {sample_mfcc.shape}   (expected: (49, {NUM_MFCC}))")

Sample label: no
Waveform shape: (16000,)
MFCC feature shape: (49, 40)   (expected: (49, 40))


**Step 5: Build `tf.data` Pipelines**

In [15]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

def make_dataset(samples, batch_size=BATCH_SIZE, shuffle=False):
    paths = [s[0] for s in samples]
    labels = [LABEL_TO_INDEX[s[1]] for s in samples]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=42, reshuffle_each_iteration=True)

    def _load_and_extract(path, label):
        waveform = load_wav(path)
        mfcc = compute_mfcc(waveform)
        mfcc = tf.expand_dims(mfcc, -1)          # (49, 40, 1)
        return mfcc, label

    ds = ds.map(_load_and_extract, num_parallel_calls=AUTOTUNE).cache()
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_samples_balanced, shuffle=True)
val_ds   = make_dataset(val_samples_balanced, shuffle=False)
test_ds  = make_dataset(test_samples_balanced, shuffle=False)

for features, labels in train_ds.take(1):
    print("Batch feature shape:", features.shape)
    print("Batch label shape:  ", labels.shape)

Batch feature shape: (64, 49, 40, 1)
Batch label shape:   (64,)


**Step 6: Define the DS-CNN-S Model**

DS-CNN-S-style architecture following Zhang et al., "Hello Edge: Keyword
Spotting on Microcontrollers" (2018): one standard convolution followed by
four depthwise-separable convolution blocks, global average pooling, and a
dense softmax head. This is a faithful reconstruction from the paper's
described structure rather than the authors' original code, so we check the
resulting parameter count below against the paper's reported ~24K for
DS-CNN-S — if it's off by a lot, the channel widths below can be adjusted.

In [16]:
from tensorflow.keras import layers, models

def build_ds_cnn_s(input_shape=(49, 40, 1), num_classes=12):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, kernel_size=(10, 4), strides=(2, 2),
                       padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for _ in range(4):
        x = layers.DepthwiseConv2D(kernel_size=(3, 3), strides=(1, 1),
                                    padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(64, kernel_size=(1, 1), strides=(1, 1),
                           padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name="DS-CNN-S")

model = build_ds_cnn_s(input_shape=(49, NUM_MFCC, 1), num_classes=len(LABELS))
model.summary()

Model: "DS-CNN-S"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 49, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 25, 20, 64)     │         2,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 25, 20, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 20, 64)     │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 25, 20, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 25, 20, 64)     │         4,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 25, 20, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 25, 20, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 25, 20, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 25, 20, 64)     │         4,096 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 24,332 (95.05 KB)

 Trainable params: 23,180 (90.55 KB)

 Non-trainable params: 1,152 (4.50 KB)

**Step 7: Compile and Train**

Adam optimizer, sparse categorical cross-entropy, with checkpointing on
best validation accuracy, early stopping, and LR reduction on plateau. This
is a plain (non-augmented) training run — mixing in background noise and
random time-shifts is a standard KWS augmentation that can be added later
if there's time, but isn't required for a first defensible baseline.

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "ds_cnn_s_best.keras", monitor="val_accuracy",
        save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.5, patience=4, verbose=1),
]

EPOCHS = 40
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

Epoch 1/40


<HOME>\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 680ms/step - accuracy: 0.6716 - loss: 1.1599
Epoch 1: val_accuracy improved from None to 0.82984, saving model to ds_cnn_s_best.keras

Epoch 1: finished saving model to ds_cnn_s_best.keras
577/577 ━━━━━━━━━━━━━━━━━━━━ 431s 742ms/step - accuracy: 0.6716 - loss: 1.1599 - val_accuracy: 0.8298 - val_loss: 0.8284 - learning_rate: 0.0010
Epoch 2/40
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.8737 - loss: 0.4337
Epoch 2: val_accuracy improved from 0.82984 to 0.87553, saving model to ds_cnn_s_best.keras

Epoch 2: finished saving model to ds_cnn_s_best.keras
577/577 ━━━━━━━━━━━━━━━━━━━━ 69s 119ms/step - accuracy: 0.8737 - loss: 0.4337 - val_accuracy: 0.8755 - val_loss: 0.3866 - learning_rate: 0.0010
Epoch 3/40
577/577 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.9027 - loss: 0.3170
Epoch 3: val_accuracy did not improve from 0.87553
577/577 ━━━━━━━━━━━━━━━━━━━━ 75s 129ms/step - accuracy: 0.9027 - loss: 0.3170 - val_accuracy: 0.8654 - val_loss: 0.4

**Step 8: Evaluate on the Test Set**

Note on the accuracy target: the roadmap earlier assumed ~94-96% based on
commonly cited DS-CNN-S figures, but the literature isn't actually
consistent on this — Cerutti et al. 2022 cite an 84.6% Hello Edge DS-CNN
baseline for the 12-class task, while other reproductions report figures in
the low-to-mid 90s, depending on training recipe and augmentation. Report
whatever this run achieves honestly in the paper rather than treating one
number as the target to hit.

In [18]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"\nFP32 baseline test accuracy: {test_accuracy*100:.2f}%")
print(f"FP32 baseline test loss:     {test_loss:.4f}")

77/77 ━━━━━━━━━━━━━━━━━━━━ 40s 515ms/step - accuracy: 0.9218 - loss: 0.3297

FP32 baseline test accuracy: 92.18%
FP32 baseline test loss:     0.3297


**Step 9: Save the FP32 Baseline Model**

This saved model is the input to the next notebook (Day 4): exporting the
four PTQ variants (FP32 reference, float16, dynamic-range INT8,
full-integer INT8) and importing each into X-CUBE-AI for the Nucleo-F446RE.

In [19]:
model.save("ds_cnn_s_fp32.keras")
print("Saved Keras model to ds_cnn_s_fp32.keras -- this is the FP32 baseline "
      "used in the next notebook (Day 4: exporting the 4 PTQ variants).")

Saved Keras model to ds_cnn_s_fp32.keras -- this is the FP32 baseline used in the next notebook (Day 4: exporting the 4 PTQ variants).


**Step 10: Reload the FP32 Baseline**

Safe to run this even if you restarted the kernel since Step 9 -- it just
loads the `.keras` file back in rather than assuming `model` is still in
memory. If you're continuing in the *same* kernel session right after Step
9, this is a no-op (just re-loads what's already there).

In [20]:
import tensorflow as tf

model = tf.keras.models.load_model("ds_cnn_s_fp32.keras")
print("Loaded FP32 baseline. Input shape:", model.input_shape)
print("Total parameters:", model.count_params())

Loaded FP32 baseline. Input shape: (None, 49, 40, 1)
Total parameters: 24332


**Step 11: Export FP32 Reference `.tflite`**

An unquantized `.tflite` conversion of the same model -- not a "PTQ
variant" by itself, but useful as an apples-to-apples reference point (same
file format, same interpreter, same evaluation code path as the three PTQ
variants below) for the accuracy comparison in Table I.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_fp32 = converter.convert()

with open("ds_cnn_s_fp32.tflite", "wb") as f:
    f.write(tflite_fp32)

print(f"FP32 .tflite size: {len(tflite_fp32):,} bytes ({len(tflite_fp32)/1024:.1f} KB)")

**Step 12: PTQ Variant 1 -- float16**

Weights are stored as float16; activations are still computed in float at
run time. No calibration data needed -- this is the simplest PTQ scheme,
and typically the safest for accuracy since it barely perturbs the
computation, at the cost of the smallest footprint reduction of the three.

In [22]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter.convert()

with open("ds_cnn_s_fp16.tflite", "wb") as f:
    f.write(tflite_fp16)

print(f"float16 .tflite size: {len(tflite_fp16):,} bytes ({len(tflite_fp16)/1024:.1f} KB)")

INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmprnythv_4\assets


INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmprnythv_4\assets


Saved artifact at '<HOME>\AppData\Local\Temp\tmprnythv_4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 40, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2892384317904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384321360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317520: 

**Step 13: PTQ Variant 2 -- Dynamic-Range INT8**

Weights are quantized to INT8; activations stay in float32 and are
quantized/dequantized on the fly at each layer during inference. Still no
calibration data needed. Smaller than float16, but the per-layer
quantize/dequantize overhead means it doesn't get the full INT8 latency
benefit that full-integer quantization does.

In [23]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dr_int8 = converter.convert()

with open("ds_cnn_s_dr_int8.tflite", "wb") as f:
    f.write(tflite_dr_int8)

print(f"dynamic-range INT8 .tflite size: {len(tflite_dr_int8):,} bytes ({len(tflite_dr_int8)/1024:.1f} KB)")

INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmpmytpzadm\assets


INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmpmytpzadm\assets


Saved artifact at '<HOME>\AppData\Local\Temp\tmpmytpzadm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 40, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2892384317904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384321360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317520: 

**Step 14: PTQ Variant 3 -- Full-Integer INT8**

Both weights *and* activations are INT8, calibrated against a
representative sample of real inputs (drawn here from `train_ds`, never
from `test_ds` -- calibrating on test data would leak test-set statistics
into the quantization ranges). `inference_input_type`/`inference_output_type`
are set to `tf.int8` so the model expects and returns raw INT8 tensors,
matching what X-CUBE-AI will generate C code for on the Nucleo board. This
is the variant with the largest expected footprint/latency win and the
largest expected accuracy risk.

In [24]:
import numpy as np

def representative_dataset():
    # ~300 single-sample calibration inputs drawn from the training split.
    for features, _ in train_ds.unbatch().batch(1).take(300):
        yield [tf.cast(features, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_full_int8 = converter.convert()

with open("ds_cnn_s_full_int8.tflite", "wb") as f:
    f.write(tflite_full_int8)

print(f"full-integer INT8 .tflite size: {len(tflite_full_int8):,} bytes ({len(tflite_full_int8)/1024:.1f} KB)")

INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmpk8kpbjiz\assets


INFO:tensorflow:Assets written to: <HOME>\AppData\Local\Temp\tmpk8kpbjiz\assets


Saved artifact at '<HOME>\AppData\Local\Temp\tmpk8kpbjiz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 40, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  2892384317904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384318288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384321360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384319824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384320208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2892384317520: 

<HOME>\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


full-integer INT8 .tflite size: 48,248 bytes (47.1 KB)


**Step 15: Desktop Accuracy Evaluation -- All Four Variants**

Runs each `.tflite` file through the TensorFlow Lite interpreter over the
*entire* test set (the same `test_ds` used for the FP32 Keras evaluation in
Step 8), so all four accuracy numbers in Table I come from the same test
samples and the same evaluation code. This handles both float-I/O variants
(FP32, float16, dynamic-range INT8) and the INT8-I/O full-integer variant
automatically, by checking each model's input dtype and quantization
parameters.

This is the accuracy figure that goes in the paper -- X-CUBE-AI's
Validate-on-target step (Day 5) re-checks agreement on only a handful of
samples actually run on the MCU; it is not a substitute for this full-test-set
number.

In [25]:
def evaluate_tflite(tflite_path, dataset):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    is_input_quantized = input_details["dtype"] in (np.int8, np.uint8)
    in_scale, in_zero_point = input_details["quantization"] if is_input_quantized else (1.0, 0)

    correct, total = 0, 0
    for batch_features, batch_labels in dataset:
        batch_features = batch_features.numpy()
        batch_labels = batch_labels.numpy()
        for i in range(batch_features.shape[0]):
            x = batch_features[i:i + 1]
            if is_input_quantized:
                x = np.round(x / in_scale + in_zero_point).astype(input_details["dtype"])
            else:
                x = x.astype(input_details["dtype"])

            interpreter.set_tensor(input_details["index"], x)
            interpreter.invoke()
            out = interpreter.get_tensor(output_details["index"])[0]
            pred = int(np.argmax(out))
            correct += int(pred == int(batch_labels[i]))
            total += 1

    accuracy = correct / total
    print(f"{tflite_path:28s}: {accuracy*100:6.2f}%  ({correct}/{total})")
    return accuracy

variant_files = {
    "FP32 (reference .tflite)": "ds_cnn_s_fp32.tflite",
    "PTQ float16": "ds_cnn_s_fp16.tflite",
    "PTQ dynamic-range INT8": "ds_cnn_s_dr_int8.tflite",
    "PTQ full-integer INT8": "ds_cnn_s_full_int8.tflite",
}

desktop_accuracy = {}
for name, path in variant_files.items():
    desktop_accuracy[name] = evaluate_tflite(path, test_ds)

print("\nSummary (desktop-evaluated accuracy, full test set):")
for name, acc in desktop_accuracy.items():
    print(f"  {name:28s}: {acc*100:6.2f}%")

<HOME>\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


ds_cnn_s_fp32.tflite        :  92.18%  (4506/4888)
ds_cnn_s_fp16.tflite        :  92.23%  (4508/4888)
ds_cnn_s_dr_int8.tflite     :  92.57%  (4525/4888)
ds_cnn_s_full_int8.tflite   :  92.25%  (4509/4888)

Summary (desktop-evaluated accuracy, full test set):
  FP32 (reference .tflite)    :  92.18%
  PTQ float16                 :  92.23%
  PTQ dynamic-range INT8      :  92.57%
  PTQ full-integer INT8       :  92.25%
